<a href="https://colab.research.google.com/github/CardiacModelling/summer-school-2025/blob/main/IonChannelModelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on session 1: ion channel models

In this session, we will

* Download and correct four ion channel models of the potassium current $\text{I}_\text{Kr}$
* Define a voltage-clamp protocol, and simulate voltage-clamp experiments with each model
* Introduce a drug-binding component and simulate channel block.


## Step 1: Installation and downloads

In [ ]:
# Don't show the installation logs
%%capture
# If installation fails, comment out the line above and run again!

# Install Myokit and Sundials
!apt-get install libsundials-dev
!pip install myokit

# Disable debug messages annoyingly shown on colab
import logging
logging.disable(logging.INFO)

# Import Myokit, matplotlib and numpy
import myokit
from matplotlib import pyplot as plt
import numpy as np

!wget https://raw.githubusercontent.com/CardiacModelling/summer-school-2025/refs/heads/main/resources/single_gate_HH.mmt
!wget https://raw.githubusercontent.com/CardiacModelling/summer-school-2025/refs/heads/main/resources/two_gate_HH.mmt
!wget https://raw.githubusercontent.com/CardiacModelling/summer-school-2025/refs/heads/main/resources/square_MM.mmt
!wget https://raw.githubusercontent.com/CardiacModelling/summer-school-2025/refs/heads/main/resources/drug_bind_MM.mmt

After running the above code block, the following model files should appear in the **Files** section of your Colab session.

```
single_gate_HH.mmt
two_gate_HH.mmt
square_MM.mmt
drug_bind_MM.mmt
```

If they don't, you can download them from the [resources](https://github.com/CardiacModelling/summer-school-2025/tree/main/resources) directory of this course's GitHub repository and manually add them to your Colab session.

## Step 2: A single-gate Hodgkin-Huxley model

The first ion channel model we will consider is a single Hodgkin-Huxley activation gate '$O$' defined by the following equation

\begin{align}
\frac{dO}{dt} = \alpha(V_m)(1-O)-\beta(V_m)O
\end{align}

where $O$ is the Open state probability, $V_m$ is the the transmembrane potential, and $\alpha$ and $\beta$ are voltaged-dependent transition rates.

✏️ **Task:** Open the `single_gate_HH.mmt` model in Colab, by double-clicking on it in the Files section, and inspect the code.

Next, we load it into Python using:

In [ ]:
single_gate_model = myokit.load_model('single_gate_HH.mmt')

## Step 3: Defining a voltage-clamp protocol

Next, we need a voltage-clamp protocol that defines the sequence of voltages we will clamp $V_m$ to.
Here, we will do this by creating a `myokit.Protocol` object, and calling `add_step(voltage, time)` for each step in the sequence.

In [ ]:
# Create a Myokit protocol with three voltage steps
protocol = myokit.Protocol()
protocol.add_step(-80, 1000)  # start at -80mV for 1s
protocol.add_step(20, 1000)   # step to 20mV for 1s
protocol.add_step(-120, 4000) # drop down to -120mV for 1s

# Define the times to simulate over
max_time = 3000 # maximum time to simulate over (3s)
dt = 0.1        # logging interval

# Create a numpy array
times = np.arange(0, max_time, dt) 

# Obtained the voltages at the specified times
Vm = protocol.value_at_times(times)

# Plot the voltage protocol
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(times, Vm)
ax.set_xlim(0, max_time)
ax.set_ylabel('Voltage (mV)')
ax.set_xlabel('Time (ms)')
plt.show()

💡 See the [Myokit documentation](https://myokit.readthedocs.io/en/stable/api_simulations/Protocol.html) for further details on Myokit Protocols, including alternative methods to create them.

## Step 4: Simulating the open probability

Next, we combine the loaded `model` and `protocol` in a `Simulation` object, and use `Simulation.run(duration)` to run a simulation.

In [ ]:
# Initialise myokit simulation with voltage protocol
single_gate_sim = myokit.Simulation(single_gate_model, protocol)

# Run simulation and save output
single_gate_log = single_gate_sim.run(max_time, log_times=times)

💡 See the [Myokit documentation](https://myokit.readthedocs.io/en/stable/api_simulations/Simulation.html#) for more information on Myokit Simulations.

Now we can plot the simulated open state probability over time.

To do this, we first look at the model code to see that the variable rerepsenting open probability is called `O` and located in the component `ikr`.
We can access it using the notation `ikr.O`:

In [ ]:
# Plot the simulated Open state probability
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6))

ax1.plot(single_gate_log.time(), single_gate_log['membrane.V'])
ax1.set_xlim(0, max_time)
ax1.set_ylabel('Voltage (mV)')

ax2.plot(single_gate_log.time(), single_gate_log['ikr.O'])
ax2.set_xlim(0, max_time)
ax2.set_ylabel('Probability')
ax2.set_xlabel('Time (ms)')

plt.tight_layout()
plt.show()

As the voltage steps up from -80mV to 0mV, the open channel probability increases from 0 to 1.
When the voltage drops down to -120mV the Open probability goes back to 0 as the channels close.

## Step 5: The ionic current
We can use the following equation to calculate the channel current based on the Open proportion

\begin{align}
I = g \cdot O \cdot (V_m - E)
\end{align}

where $g$ is the channel conductance, $V_m$ is transmembrane potential, and $E$ is the Nernst potential, calculated as

\begin{align}
E = \frac{RT}{F}\log \left( \frac{K_o}{K_i} \right)
\end{align}

where $R$ is the universal ideal gas constant, $T$ is temperature (in Kelvin), $F$ is Faraday's constant, and $K_o$ and $K_i$ are the extracellular and intracellular Potassium concentrations respectively.

✏️ **Task:** Add the equation for the Nernst potential to the model in `single_gate_HH.mmt` (near the bottom of the file), and the equation for the current (near the top).

💡 The values for $R$, $T$, and $F$ and both concentrations are already filled in near the bottom of the file.

💡 The correct current should reach an equilibrium of around 17 pA during the 0mV step.

In [ ]:
# Re-load the model and create & run a new simulation
single_gate_model = myokit.load_model('single_gate_HH.mmt')
single_gate_sim = myokit.Simulation(single_gate_model, protocol)
single_gate_log = single_gate_sim.run(max_time, log_times=times)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(single_gate_log.time(), single_gate_log['ikr.IKr'])
ax.set_xlim(0, max_time)
ax.set_ylabel('Current (pA)')
ax.set_xlabel('Time (ms)')
plt.show()

Note how the initial part is a simple scaling from the open probability, but with the step back down to -120mV it becomes more interesting, as $(V_m - E)$ changes sign.

## Step 6: A two-gate Hodgkin-Huxley model

We will now increase the model complexity by introducing a _recovery gate_, `r`

This is partially implemented in the file `two_gate_HH.mmt`.
This reuses reuses the open state from the previous code as _activation gate_, renamed `a` for clarity.
But the equations for the recovery gate are missing.

✏️ **Task:** Complete the model equation for the recovery gate in `two_gate_HH.mmt`, and run the code below

💡 In the model file, the transition rate for $(1 - r) \rightarrow r$ is called `alpha_r`, and the rate for $r \rightarrow (1 - r)$ is called `beta_r`.

💡 As an initial value for $r$, you can use 1.

Once you have completed the missing ODE, run the below code block to load and run the two gate Hodgkin-Huxley model simulation.

In [ ]:
two_gate_model = myokit.load_model('two_gate_HH.mmt')
two_gate_sim = myokit.Simulation(two_gate_model, protocol)
two_gate_log = two_gate_sim.run(max_time, log_times=times)

In the two-gate model, the channel is defined as being open only when both of the independent gates are in an 'open' configuration.
Therefore, we can calculate the open state probability by taking the product of `a` and `r`.

In [ ]:
# Plot the simulated open state probability a * r
two_gate_O = np.array(two_gate_log['ikr.a']) * np.array(two_gate_log['ikr.r'])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(times, two_gate_O)
ax.set_xlim(0, max_time)
ax.set_title('Open State Probability')
ax.set_ylabel('Probability')
ax.set_xlabel('Time (ms)')
plt.show()

✏️ **Task:** Complete the model equation for `I` in `two_gate_HH.mmt` and run the code below to plot the current.

In [ ]:
# Plot the current directly
two_gate_model = myokit.load_model('two_gate_HH.mmt')
two_gate_sim = myokit.Simulation(two_gate_model, protocol)
two_gate_log = two_gate_sim.run(max_time, log_times=times)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(two_gate_log.time(), two_gate_log['ikr.IKr'])
ax.set_xlim(0, max_time)
ax.set_title('Current')
ax.set_ylabel('Current (pA)')
ax.set_xlabel('Time (ms)')
plt.show()

## Step 7: The equivalent four-state Markov model

We now consider a four-state Markov model that is equivalent to the previous two-gate Hodgkin-Huxley model.

Its four model states are:

* Closed, "C", similar to $a=0, r=1$
* Open, "O", similar to $a=1, r=1$
* Closed inactive, "CI", similar to $a=0, r=0$
* Inactive, "I", similar to $a=1, r=0$

Double-click on the `square_MM.mmt` file to inspect the Myokit model file.
Note that the ODE for the open state is currently incomplete.

✏️ **Task:** Complete the model equation for the open state by editing `square_MM.mmt`.

💡 With pen & paper, draw a diagram of the states and rates. Start with "Closed" and "Open" on the top row, representing the "recovered" states. Then add the "Closed inactivated" and "Inactivated" states on the bottom row and add the rates $\alpha_a$, $\beta_a$, $\alpha_r$ and $\beta_r$

💡 The `alpha_a` rate governs the transition from C to O, but also from CI to I. All other rates also appear twice.

💡 To reduce computational cost, we only model 3 out of 4 states, and use the fact that all occupancies must add up to 1 to calculate the 4th.

Once you have completed the missing ODE, run the below code block to load and run the four-state Markov model simulation.

In [ ]:
square_model = myokit.load_model('square_MM.mmt')
square_sim = myokit.Simulation(square_model, protocol)
square_log = square_sim.run(max_time, log_times=times)

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(0, max_time)
ax.set_ylabel('Current (pA)')
ax.set_xlabel('Time (ms)')
ax.plot(two_gate_log.time(), two_gate_log['ikr.IKr'], label='HH current')
ax.plot(square_log.time(), square_log['ikr.IKr'], ls='--', label='Markov current')
plt.legend()
plt.show()

Both models should now produce exactly the same output.

## Step 8: Add a drug-binding state to the Markov model

Above, you have shown how the more complex Markov model produced exactly the same output as the simpler Hodgkin-Huxley model.
We now extend it in a way that's not possible for the HH model, by adding drug binding _only from the open (activated and recovered) state_.

To do this, we add a 5th state, "OD", and connect it only to the open state "O".
The transitions into and out of this state won't be voltage depedent, but instead depend on a drug concentration, $[D]$, a Hill coefficient, $h$, and on/off binding rates, $k_{on}$ and $k_{off}$.
The full differential equation for OD is:

\begin{align}
\frac{dOD}{dt} = k_{on}[D]^h\cdot O - k_{off}\cdot OD
\end{align}

Double-click on the `drug_bind_MM.mmt` file to inspect the Myokit model file.
This model contains the on/off rates and Hill coefficient, but not the new equation for OD.

✏️ **Task:** Complete the model by adding the equation for OD and making any other required changes

💡 Because OD interacts with O, the equation for O will need to be updated as well

💡 Is the equation for CI still correct?

Once you have modified the model, run the below code block to load and run the drug binding Markov model simulation.
This code runs one simulation without a drug ($[D]=0$) and then uses `reset()` to reset the simulation before repeating with a drug compound added ($[D] = 1500$).

In [ ]:
# Load the model, create a simulation
drug_bind_model = myokit.load_model('drug_bind_MM.mmt')
drug_bind_sim = myokit.Simulation(drug_bind_model, protocol)

# Run with the default concentration of 0
drug_bind_log_1 = drug_bind_sim.run(max_time, log_times=times)

# Reset, and re-run with an updated concentration
drug_bind_sim.reset()
drug_bind_sim.set_constant('ikr.D', 1500)
drug_bind_log_2 = drug_bind_sim.run(max_time, log_times=times)

# Compare the results
fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(0, max_time)
ax.set_ylabel('Current (pA)')
ax.set_xlabel('Time (ms)')
ax.plot(drug_bind_log_1.time(), drug_bind_log_1['ikr.IKr'], label='No Drug')
ax.plot(drug_bind_log_2.time(), drug_bind_log_2['ikr.IKr'], label='Drug')
plt.legend()
plt.show()

# Extra tasks

✏️ Return to Step 3 and make changes to the voltage protocol.
Observe how this affects the simulated Open proportion and current for each model.

✏️ Adjust the model parameters in the `.mmt` files of each model and observe how they influence the model dynamics.

✏️ Add an additional ID (Inactive Drug-bound) state and I-ID drug binding dynamics to the `drug_bind_MM.mmt` model file and observe the effects.